In [1]:
import argparse

import matplotlib.pyplot as plt

from data import *
from utils import *
from datetime import datetime

In [2]:
parser = argparse.ArgumentParser(description='Gen-RKM Model')

parser.add_argument('--N', type=int, default=100, help='Total # of samples')
parser.add_argument('--mb_size', type=int, default=100, help='Mini-batch size. See utils.py')
parser.add_argument('--h_dim', type=int, default=10, help='Dim of latent vector')
parser.add_argument('--capacity', type=int, default=32, help='Capacity of network. See utils.py')
parser.add_argument('--x_fdim', type=int, default=128, help='Input x_fdim. See utils.py')
parser.add_argument('--y_fdim', type=int, default=20, help='Input y_fdim. See utils.py')
parser.add_argument('--c_accu', type=float, default=100, help='Input weight on recons_error')

# Training Settings =============================
parser.add_argument('--lr', type=float, default=1e-4, help='Input learning rate for optimizer')
parser.add_argument('--max_epochs', type=int, default=1000, help='Input max_epoch for cut-off')
parser.add_argument('--device', type=str, default='cpu', help='Device type: cuda or cpu')
parser.add_argument('--workers', type=int, default=0, help='# of workers for dataloader')
parser.add_argument('--shuffle', type=bool, default=False, help='shuffle dataset: true or false')

opt, _ = parser.parse_known_args()

In [3]:
xtrain, ipVec_dim, nChannels = get_mnist_dataloader(args=opt)

In [4]:
ct = time.strftime("%Y%m%d-%H%M")
dirs = create_dirs(ct=ct)
dirs.create()

In [5]:
def kPCA(X, Y):
    a = torch.mm(X, torch.t(X)) + torch.mm(Y, torch.t(Y))
    nh1 = a.size(0)
    oneN = torch.div(torch.ones(nh1, nh1), nh1).double().to(opt.device)
    a = a - torch.mm(oneN, a) - torch.mm(a, oneN) + torch.mm(torch.mm(oneN, a), oneN)  # centering
    h, s, _ = torch.svd(a, some=False)
    return h[:, :opt.h_dim], s

In [6]:
def rkm_loss(output1, X, output2, Y):
    h, s = kPCA(output1, output2)
    U = torch.mm(output1.t(), h)
    V = torch.mm(output2.t(), h)

    x_tilde = net3(torch.mm(h, U.t()))
    y_logits = net4(torch.mm(h, V.t()))   # treat as logits (no softmax here)

    # Costs
    f1 = torch.trace(torch.mm(torch.mm(output1, U), h.t())) + torch.trace(
         torch.mm(torch.mm(output2, V), h.t()))
    f2 = 0.5 * torch.trace(torch.mm(h, torch.mm(torch.diag(s[:opt.h_dim]), h.t())))
    f3 = 0.5 * (torch.trace(torch.mm(U.t(), U)) + torch.trace(torch.mm(V.t(), V)))

    # Reconstruction losses
    mse = torch.nn.MSELoss()
    img_loss = mse(x_tilde.view(-1, ipVec_dim), X.view(-1, ipVec_dim))

    # Label loss: supports Y as one-hot (B,10) or indices (B,)
    if Y.dim() == 2 and Y.size(-1) == 10:
        y_idx = Y.argmax(dim=1).long()
    else:
        y_idx = Y.long().view(-1)

    ce = torch.nn.CrossEntropyLoss()
    lbl_loss = ce(y_logits.view(-1, 10), y_idx)

    f4 = img_loss + lbl_loss

    loss = -f1 + f3 + f2 + 0.5 * (-f1 + f3 + f2) ** 2 + opt.c_accu * f4
    return loss

In [7]:
net1 = Net1().double().to(opt.device)
net2 = Net2().double().to(opt.device)
net3 = Net3().double().to(opt.device)
net4 = Net4().double().to(opt.device)

In [8]:
params = list(net1.parameters()) + list(net3.parameters()) + list(net2.parameters()) + list(net4.parameters())
optimizer = torch.optim.Adam(params, lr=opt.lr, weight_decay=0)

In [9]:
l_cost = 6  # Costs from where checkpoints will be saved
t = 1
cost = np.inf  # Initialize cost
start = datetime.now()
while cost > 0.2 and t <= opt.max_epochs:  # run epochs until convergence
    avg_loss = 0
    for i, (datax, datay) in enumerate(xtrain):
        if i < math.ceil(opt.N / opt.mb_size):
            datax, datay = datax.to(opt.device), datay.to(opt.device)
            output1 = net1(datax)
            output2 = net2(datay)
            loss = rkm_loss(output1, datax, output2, datay)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            avg_loss += loss.detach().cpu().numpy()
        else:
            break
    cost = avg_loss

    # Remember lowest cost and save checkpoint
    is_best = cost < l_cost
    l_cost = min(cost, l_cost)

    dirs.save_checkpoint({
        'epochs': t + 1,
        'net1_state_dict': net1.state_dict(),
        'net3_state_dict': net3.state_dict(),
        'net2_state_dict': net2.state_dict(),
        'net4_state_dict': net4.state_dict(),
        'l_cost': l_cost,
        'optimizer': optimizer.state_dict()}, is_best)
    print(t, cost)
    t += 1
print('Finished Training in: {}. Lowest cost: {}'.format(str(datetime.now() - start), l_cost))

1 254.02590888689267
2 253.93518248244303
3 253.8468434621452
4 253.75831844094427
5 253.67039228849
6 253.582369197602
7 253.4934237496944
8 253.40288511763006
9 253.31256805185512
10 253.27060574313882
11 253.14918989007134
12 253.06804588003098
13 252.98792726127962
14 252.9038616980297
15 252.82206555586123
16 252.77873381772127
17 252.6704532347734
18 252.58824673233804
19 252.51184229413118
20 252.43486080720118
21 252.3550179966951
22 252.2724841028003
23 252.1880583871527
24 252.1015410559075
25 252.01353854450372
26 251.92519264725146
27 251.85748037865085
28 251.86574051374
29 251.76473878509543
30 251.67521207417727
31 251.59262014975528
32 251.51288772557635
33 251.43444313795274
34 251.35593132537542
35 251.27710717382743
36 251.19768951267403
37 251.11740025280986
38 251.03608058663403
39 250.9532398587734
40 250.86864899555823
41 250.78214050164004
42 250.6939026662623
43 250.6048430690506
44 250.5148696998267
45 250.42768580801481
46 250.41320210349076
47 250.2645243008

In [10]:
if os.path.exists('cp/{}'.format(dirs.dircp)):
    sd_mdl = torch.load('cp/{}'.format(dirs.dircp))
    net1.load_state_dict(sd_mdl['net1_state_dict'])
    net3.load_state_dict(sd_mdl['net3_state_dict'])
    net2.load_state_dict(sd_mdl['net2_state_dict'])
    net4.load_state_dict(sd_mdl['net4_state_dict'])

U, V, h, s = final_compute(opt, net1, net2, kPCA)

/opt/anaconda3/envs/MT-GenRKM/lib/python3.12/site-packages/torchvision/datasets/mnist.py:76: UserWarning: train_data has been renamed data
  warnings.warn("train_data has been renamed data")


In [11]:
torch.save({'net1': net1,
            'net3': net3,
            'net2': net2,
            'net4': net4,
            'net1_state_dict': net1.state_dict(),
            'net3_state_dict': net3.state_dict(),
            'net2_state_dict': net2.state_dict(),
            'net4_state_dict': net4.state_dict(),
            'h': h,
            'opt': opt, 's': s, 'U': U, 'V': V}, 'out/{}'.format(dirs.dirout))